In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import pandas as pd

clean_results_path = "../clean_results/res_clean.json"
dirty_results_path = "../logs/silent-norm-ablations-v2/results.json"

clean_results = pd.read_json(clean_results_path, orient="records")
dirty_results = pd.read_json(dirty_results_path, orient="records")

display(clean_results.sample())
display(dirty_results.sample())

In [ ]:
def parse_path_clean(path: str) -> dict:
    parts = path.split("/")
    model_name = parts[-2]
    train_dataset = None
    layer_name = None
    exp_name = None

    return {"model_name": model_name, "train_dataset": train_dataset, "layer_name": layer_name, "exp_name": exp_name}


def parse_path_dirty(path: str) -> dict:
    parts = path.split("/")
    model_name = parts[-5]
    train_dataset = parts[-4]
    layer_name = parts[-3]
    exp_name = parts[-2]

    return {"model_name": model_name, "train_dataset": train_dataset, "layer_name": layer_name, "exp_name": exp_name}


# # now use parse_path to add corresponding columns to the dataframe
# parsed = results["path"].apply(parse_path)
# results = pd.concat([results, parsed.apply(pd.Series)], axis=1)

# # results = results.drop(columns=["path", "file"])

# results.sample(20)

parsed = clean_results.apply(lambda row: pd.Series(parse_path_clean(row["path"])), axis=1)
clean_results = pd.concat([clean_results, parsed], axis=1)


parsed = dirty_results.apply(lambda row: pd.Series(parse_path_dirty(row["path"])), axis=1)
dirty_results = pd.concat([dirty_results, parsed], axis=1)

display(clean_results.sample())
display(dirty_results.sample())

In [ ]:
def parse_experiment(exp_name: str) -> dict:
    kl_value = None
    lr_value = None
    early_stop = True

    if "no-early-stop" in exp_name:
        early_stop = False

    if "baseline" in exp_name or "no-early-stop" in exp_name:
        lr_value = 0.1
        kl_value = 1.0

    if "small-lr" in exp_name:
        lr_value = 0.02
        kl_value = 1.0

    if "medium-lr" in exp_name:
        lr_value = 0.04
        kl_value = 1.0

    if "large-lr" in exp_name:
        lr_value = 0.25
        kl_value = 1.0

    if "small-kl" in exp_name:
        lr_value = 0.1
        kl_value = 0.5

    if "high-kl" in exp_name:
        lr_value = 0.1
        kl_value = 2.0

    # otherwise, the kl should appear as .._kl=<number>-..

    if "kl=" in exp_name:
        lr_value = 0.1

        kl_part = [part for part in exp_name.split("_") if part.startswith("kl=")]
        if not kl_part:
            raise ValueError(f"Could not find kl value in experiment name: {exp_name}")

        kl_value_str = kl_part[0].split("=")[1].split("-")[0]
        kl_value = float(kl_value_str)

    if lr_value is None or kl_value is None:
        raise ValueError(f"Could not parse experiment name: {exp_name}")

    return {"kl_value": kl_value, "lr_value": lr_value, "early_stop": early_stop}


parsed_exp = dirty_results["exp_name"].apply(lambda x: pd.Series(parse_experiment(x)))
dirty_results = pd.concat([dirty_results, parsed_exp], axis=1)

display(dirty_results.sample())


In [ ]:
def intersect_on_columns(df1: pd.DataFrame, df2: pd.DataFrame, on: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Filter two dataframes to only keep rows whose values in `on` columns
    appear in both dataframes. Rows whose key is found only in one dataframe
    are discarded.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Input dataframes to filter.
    on : list[str]
        Column name(s) to use as the intersection key.

    Returns
    -------
    (df1_filtered, df2_filtered) : tuple of pd.DataFrame
    """
    common_keys = pd.merge(
        df1[on].drop_duplicates(),
        df2[on].drop_duplicates(),
        on=on,
    )
    df1_filtered = df1.merge(common_keys, on=on)
    df2_filtered = df2.merge(common_keys, on=on)
    return df1_filtered, df2_filtered


In [ ]:
print(dirty_results["kl_value"].unique())
print(dirty_results["lr_value"].unique())
print(dirty_results["early_stop"].unique())
dirty_results["train_dataset"].unique()

In [ ]:
# format metric names

def fmt_metric_name(metric_name: str) -> str:
    metric_name = metric_name.replace(",none", "")
    return metric_name

clean_results["metric"] = clean_results["metric"].apply(fmt_metric_name)
dirty_results["metric"] = dirty_results["metric"].apply(fmt_metric_name)

# remove all metrics called blimp_ leave only pure blimp metric

def filter_metrics(df: pd.DataFrame) -> pd.DataFrame:
    to_remove = df["benchmark"].str.startswith("blimp_")
    return df[~to_remove]

clean_results = filter_metrics(clean_results)
dirty_results = filter_metrics(dirty_results)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from typing import Callable

from pyarrow import dataset

KNOWN_METRIC_KEYWORDS = ("acc", "f1", "match", "validity", "compliance", "pass")
PLOT_COLUMNS = [
    "benchmark",
    "metric",
    "benchmark_metric",
    "value_dirty",
    "value_clean",
    "value_plot",
    "is_known_range",
]


def is_known_range_metric(metric_name: str) -> bool:
    metric_name = str(metric_name).lower()
    return any(keyword in metric_name for keyword in KNOWN_METRIC_KEYWORDS)


def empty_plot_df() -> pd.DataFrame:
    return pd.DataFrame(columns=PLOT_COLUMNS)


def resolve_value_column(df: pd.DataFrame) -> str | None:
    if "value" in df.columns:
        return "value"

    fallback_candidates = [
        column_name
        for column_name in df.columns
        if column_name.startswith("value")
    ]
    if fallback_candidates:
        return fallback_candidates[0]

    return None


def prepare_plot_data(dirty_group: pd.DataFrame, clean_group: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if dirty_group.empty or clean_group.empty:
        return empty_plot_df(), empty_plot_df()

    dirty_value_col = resolve_value_column(dirty_group)
    clean_value_col = resolve_value_column(clean_group)

    if dirty_value_col is None or clean_value_col is None:
        return empty_plot_df(), empty_plot_df()

    dirty_prepared = dirty_group[["benchmark", "metric", dirty_value_col]].rename(
        columns={dirty_value_col: "value_dirty"}
    )
    clean_prepared = clean_group[["benchmark", "metric", clean_value_col]].rename(
        columns={clean_value_col: "value_clean"}
    )

    merged = dirty_prepared.merge(
        clean_prepared,
        on=["benchmark", "metric"],
        how="inner",
    ).copy()

    if merged.empty:
        return empty_plot_df(), empty_plot_df()

    merged = merged.dropna(subset=["value_dirty", "value_clean"])

    if merged.empty:
        return empty_plot_df(), empty_plot_df()

    merged["benchmark_metric"] = merged["benchmark"] + " / " + merged["metric"]
    merged["is_known_range"] = merged["metric"].apply(is_known_range_metric)

    known_df = merged[merged["is_known_range"]].copy()
    unknown_df = merged[~merged["is_known_range"]].copy()

    known_df["value_plot"] = (known_df["value_dirty"] - known_df["value_clean"]) * 100.0
    unknown_df["value_plot"] = unknown_df["value_dirty"] - unknown_df["value_clean"]

    known_df = known_df.sort_values("benchmark_metric", ascending=True).reset_index(drop=True)
    unknown_df = unknown_df.sort_values("benchmark_metric", ascending=True).reset_index(drop=True)

    return known_df, unknown_df


def annotate_bars(ax, df: pd.DataFrame, label_fmt: Callable, fontsize: int = 6) -> None:
    for idx, bar in enumerate(ax.patches):
        if idx >= len(df):
            break

        dirty_value = df.loc[idx, "value_dirty"]
        clean_value = df.loc[idx, "value_clean"]
        label = label_fmt(dirty_value=dirty_value, clean_value=clean_value)

        height = bar.get_height()
        x_center = bar.get_x() + bar.get_width() / 2
        y_anchor = max(height, 0)

        ax.annotate(
            label,
            xy=(x_center, y_anchor),
            xytext=(0, 2),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=fontsize,
        )


def plot_panel(
    ax,
    df: pd.DataFrame,
    ylabel: str,
    panel_title: str,
    label_fmt: Callable | None = None,
    fontsize: int = 6,
) -> None:
    ax.set_title(panel_title, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=90, labelsize=fontsize)
    ax.tick_params(axis="y", labelsize=fontsize)
    ax.grid(axis="y", alpha=0.3)

    colors = ["#d62728" if value < 0 else "#1f77b4" for value in df["value_plot"]]
    ax.bar(df["benchmark_metric"], df["value_plot"], color=colors)

    if label_fmt is not None:
        annotate_bars(ax, df, fontsize=fontsize, label_fmt=label_fmt)


In [31]:


dirty_results_filtered = dirty_results[
    (dirty_results["kl_value"] == 0.5)
    & (dirty_results["lr_value"] == 0.1)
    & (dirty_results["early_stop"] == True)
    # & (dirty_results["train_dataset"] == "oasst2")
] .copy()

groups = dirty_results_filtered[["model_name", "train_dataset", "layer_name"]].drop_duplicates()

for _, group in groups.iterrows():
    model_name = group["model_name"]
    train_dataset = group["train_dataset"]
    layer_name = group["layer_name"]

    dirty_group = dirty_results_filtered[
        (dirty_results_filtered["model_name"] == model_name)
        & (dirty_results_filtered["train_dataset"] == train_dataset)
        & (dirty_results_filtered["layer_name"] == layer_name)
    ]
    clean_group = clean_results[clean_results["model_name"] == model_name]

    dirty_group, clean_group = intersect_on_columns(
        dirty_group,
        clean_group,
        on=["benchmark", "metric"],
    )

    known_df, unknown_df = prepare_plot_data(dirty_group, clean_group)

    panel_specs = []
    if not known_df.empty:
        panel_specs.append(
            {
                "df": known_df,
                "ylabel": "Difference %  (for range [0-1] metrics)",
                "title": "Known-range metrics: (dirty - clean) x 100",
                "label_fmt": lambda dirty_value, clean_value: f"{(dirty_value - clean_value) * 100:.1f}",
                "fontsize": 10,
            }
        )
    if not unknown_df.empty:
        panel_specs.append(
            {
                "df": unknown_df,
                "ylabel": "Difference (raw)",
                "title": "Unknown-range metrics: (dirty - clean)",
                "label_fmt": lambda dirty_value, clean_value: f"{dirty_value:.3f} - {clean_value:.3f}",
                "fontsize": 10,
            }
        )

    if not panel_specs:
        continue

    fig, axes = plt.subplots(len(panel_specs), 1, figsize=(18, 5 * len(panel_specs)))
    if len(panel_specs) == 1:
        axes = [axes]

    fig.suptitle(
        f"Model: {model_name} | Dataset: {train_dataset} | Layer: {layer_name}",
        y=1.04,
        fontsize=11,
        # bold
        fontweight="bold",
    )

    for ax, spec in zip(axes, panel_specs):
        plot_panel(
            ax=ax,
            df=spec["df"],
            ylabel=spec["ylabel"],
            panel_title=spec["title"],
            label_fmt=spec["label_fmt"],
            fontsize=spec["fontsize"],
        )

    plt.tight_layout()
    plt.show()

    # break
